In [ ]:
import torch
from torch.utils.data import Dataset
from dataclasses import dataclass
import numpy as np
from v2_fm_pmssm.data import *

In [35]:
# parameter, parameter_table, point_id, _ = load_data('/u/yinya/archive/gen_flat_v2_test/ntuples')

branches = ['MO_Omega', 'SS_m_h']
parameter, parameter_table, point_id, _ = load_data('/u/yinya/archive/gen_flat_v2_test/ntuples', branches)

---Load pMSSM dataset v2---
Loading files: 2/2
Loaded 2 files from ['/u/yinya/archive/gen_flat_v2_test/ntuples']
Total samples: 16000


In [ ]:
@dataclass(frozen=True)
class Variable:
    branch: str             # name of the ROOT branch, e.g. "IN_M1"
    step: str               # generated by which step, e.g. "IN", "SP", ...
    data_type: str          # type of observable, e.g. "theory_parameter", "mass", "cross_section"
#     value_type: str         # data type, e.g. int, float, used to seperate categorical params
    transform: str          # "log10", "standard", ...
    invalid: tuple = ()     # value for invalid data e.g. -1 for MO_Omega

input_params = ['IN_M_1', 'IN_M_2', 'IN_M_3', 'IN_At', 'IN_Ab', 'IN_Atau', 'IN_mu', 
                'IN_tanb', 'IN_mA',  'IN_mqL3', 'IN_mtR', 'IN_mbR']

all_var = [Variable(p, "IN", "theory_parameter", "standard") for p in input_params]
all_var += [Variable("MO_Omega", "MO", "dark_matter", "log10", invalid=(-1,))]
all_var += [Variable("SS_m_h", "SS", "mass", "standard", invalid=(-1,))]

In [55]:
class pMSSM_Dataset(Dataset):
    def __init__(self, parameters, point_id, variables, device):
        self.parameters = torch.as_tensor(parameters, dtype=torch.float32, device=device)
        self.point_id = point_id
        self.variables = variables  # list[Variables], the variable registry
        self._col_nr = {v.branch: i for i, v in enumerate(variables)} # column number of given branch names
        self._log_cols = [i for i, v in enumerate(variables) if v.transform == "log10"]
        self.mean = None
        self.std = None
        self.transformed = False   # True after transform(), False after inverse()

    def __len__(self):
        return len(self.parameters)

    def __getitem__(self, key):
        if isinstance(key, str):
            return self.parameters[:, self._col_nr[key]]
        if isinstance(key, (tuple, list)) and all(isinstance(k, str) for k in key):
            idx = [self._col_nr[k] for k in key]
            return self.parameters[:, idx]
        return self.parameters[key]

    def setDevice(self, device):
        self.parameters = self.parameters.to(device)
    
    def subset(self, rows):
        """New dataset with the selected rows (bool mask or index list/array/tensor); keeps transform state."""
        rows = torch.as_tensor(rows, device=self.parameters.device)
        if rows.dtype == torch.bool and len(rows) != len(self):
            raise ValueError(f"mask has length {len(rows)}, dataset has {len(self)}")
        new = type(self)(self.parameters[rows], self.point_id[rows.cpu().numpy()],
                        self.variables, self.parameters.device)
        new.mean, new.std, new.transformed = self.mean, self.std, self.transformed
        return new

    def filter_invalid(self, branches, inplace=False):
        """Drop rows where any of the given branches has one of its invalid values."""
        if isinstance(branches, str):
            branches = [branches]
        bad = torch.zeros(len(self), dtype=torch.bool, device=self.parameters.device)
        for b in branches:
            i = self._col_nr[b]
            if self.variables[i].invalid:
                col = self.parameters[:, i]
                vals = torch.as_tensor(self.variables[i].invalid, dtype=col.dtype, device=col.device)
                bad |= torch.isin(col, vals)
        keep = ~bad
        if inplace:
            self.parameters = self.parameters[keep]
            self.point_id = self.point_id[keep.cpu().numpy()]
            return self
        return type(self)(self.parameters[keep], self.point_id[keep.cpu().numpy()],
                            self.variables, self.parameters.device)

    def transform(self, mean=None, std=None):
        """log10 where the variable asks for it, then standardize in place.
        Without mean/std: compute them from this dataset (training set).
        With mean/std: reuse them (validation/test set must use the training statistics)."""
        if self.transformed:
            raise RuntimeError("dataset is already transformed")
        x = self.parameters
        x[:, self._log_cols] = torch.log10(x[:, self._log_cols])
        if mean is None or std is None:
            mean, std = x.mean(dim=0), x.std(dim=0)
        x.sub_(mean).div_(std)
        self.mean, self.std = mean, std
        self.transformed = True
        return mean, std

    def inverse(self, mean=None, std=None):
        """Undo transform in place: de-standardize, then 10**x for log10 columns."""
        if not self.transformed:
            raise RuntimeError("dataset is not transformed")
        mean = self.mean if mean is None else mean
        std = self.std if std is None else std
        x = self.parameters
        x.mul_(std).add_(mean)
        x[:, self._log_cols] = 10 ** x[:, self._log_cols]
        self.transformed = False


In [ ]:
def select(variables, branches=None, steps=None) -> list[Variable]:
    """Pick variables by branch name and/or step; result order = column order."""
    if branches is not None:
        name_to_var = {v.branch: v for v in variables}
        return [name_to_var[b] for b in branches]          # KeyError on typos
    return [v for v in variables if steps is None or v.step in steps]

var_registry = select(all_var, branches=branches)
var_registry

[Variable(branch='MO_Omega', step='MO', data_type='dark_matter', transform='log10', invalid=(-1,)),
 Variable(branch='SS_m_h', step='SS', data_type='mass', transform='standard', invalid=(-1,))]

In [43]:
# def replace_invalid(self, inplace=True):
#     """Replace each variable's invalid values (e.g. -1) with NaN.
#     inplace=True changes this dataset; inplace=False returns a new dataset."""
#     table = self.parameters if inplace else self.parameters.clone()
#     for i, var in enumerate(self.variables):
#         if var.invalid:
#             col = table[:, i]
#             bad = torch.as_tensor(var.invalid, dtype=col.dtype, device=col.device)
#             col[torch.isin(col, bad)] = float('nan')
#     if inplace:
#         return self
#     return type(self)(table, self.point_id, self.variables, table.device)

# print(parameter_table[0])
# print(parameter['MO_Omega'])
# print(parameter_table.dtype)

# # parameter_table = replace_invalid(parameter_table, var_registry)

# print(parameter_table[0])
# print(parameter['MO_Omega'])
# print(parameter_table.dtype)


In [52]:
dataset = pMSSM_Dataset(parameter_table, point_id, var_registry, 'cpu')

In [ ]:
dataset['IN_M_1',"IN_M_2"]

tensor([[-1303.2849, -1105.5189],
        [ 1461.4100,  -788.9185],
        [ -455.2041,  1256.8882],
        ...,
        [-1587.8308,  -831.1736],
        [-1354.3394,  1273.0024],
        [ 1104.8632,   905.1461]])

In [45]:
dataset[0,1]

tensor(128.6721)

In [53]:
dataset = dataset.filter_invalid(branches=['MO_Omega'])
len(dataset)

12663

In [ ]:
def split_indices(n, fractions=(0.8, 0.1, 0.1), seed=42):
    """Randomly split range(n) into disjoint index arrays.
    Subset sizes depend only on n and fractions (never on the seed);
    the seed only decides which points go where."""
    fractions = np.asarray(fractions, dtype=float)
    if not np.isclose(fractions.sum(), 1.0):
        raise ValueError(f"fractions must sum to 1, got {fractions.sum()}")
    bounds = np.round(np.cumsum(fractions) * n).astype(int)   # end of each subset
    bounds[-1] = n                                             # guard against rounding
    perm = np.random.default_rng(seed).permutation(n)
    return [np.sort(part) for part in np.split(perm, bounds[:-1])]


In [54]:
train_idx, val_idx, test_idx = split_indices(len(dataset), (0.8, 0.1, 0.1), seed=42)
train, val, test = (dataset.subset(i) for i in (train_idx, val_idx, test_idx))
mean, std = train.transform()
val.transform(mean, std); test.transform(mean, std)


(tensor([ -0.9145, 123.0667]), tensor([ 2.0294, 17.5796]))